In [4]:
import requests as r
import json
import csv
import pandas as pd
from pandas import json_normalize

#url to post
PokeDexURL = 'https://pokeapi.co/api/v2/pokedex/1/'
#get data from that URL
res = r.get(PokeDexURL)
#PokeDex (National)

#put the data coming in from res into values as json
values = res.json()

#Define variables
PokeDex = [] 
MaxDexNumFound = 0
MaxDexNum = 0
CurrentNum = 0
PokeDexID = ''
PokemonName = ''

#Define headers
PokeDex.append(['PokeDexID|PokemonName'])

#Load table
while MaxDexNumFound == 0:
    #Check to see if PokeDexID exists 
    try:
        #If PokeDexID exists, add to extract
        PokeDexID =str(values['pokemon_entries'][CurrentNum]['entry_number'])
        PokemonName = values['pokemon_entries'][CurrentNum]['pokemon_species']['name'].title()
        PokeDex.append([PokeDexID + '|' + PokemonName])
        CurrentNum += 1
        #If PokeDexID does not exist, print the max PokeDexID found
    except:
        print('Max PokeDexID Found:' + str(CurrentNum))
        MaxDexNumFound = 1
        MaxDexNum = CurrentNum

Max PokeDexID Found:1025


In [8]:
PokeDex[0:10]

[['PokeDexID|PokemonName'],
 ['1|Bulbasaur'],
 ['2|Ivysaur'],
 ['3|Venusaur'],
 ['4|Charmander'],
 ['5|Charmeleon'],
 ['6|Charizard'],
 ['7|Squirtle'],
 ['8|Wartortle'],
 ['9|Blastoise']]

In [ ]:
#https://www.tcgplayer.com/search/pokemon/product?Rarity=Illustration+Rare|Special+Illustration+Rare&Price_Condition=Less+Than&advancedSearch=true&productLineName=pokemon&view=grid&page=1
#https://www.tcgplayer.com/search/pokemon/product?productLineName=pokemon&q=trainer+gallery&view=grid&page=1
#https://www.tcgplayer.com/search/pokemon-japan/product?productLineName=pokemon-japan&view=grid&RarityName=Art+Rare|Special+Art+Rare&page=1
#https://www.tcgplayer.com/search/pokemon-japan/product?productLineName=pokemon-japan&view=grid&RarityName=Character+Rare|Character+Super+Rare&page=1

In [ ]:
Extract = ['ProductPK','Series','CardName','CardNumber','Rarity','SpotlightPrice']
ProductPK = [['452021'],['509983'],['618701'],['6187015555555']]
BaseURL = 'https://www.tcgplayer.com/product/'
FailedPKChecks = 0

from selenium import webdriver
from selenium.webdriver.common.by import By
import time

# Set up the WebDriver
driver = webdriver.Chrome()

for PK in ProductPK:

    # Open the news website
    driver.get(BaseURL + PK[0])

    tempList = []

    # Allow the page to load
    time.sleep(5)

    print('ProductPK: ' + PK[0])
    tempList.append(PK[0])

    ############---------BreadCrumbList

    # Try searching the next PK
    try: 
        BreadCrumbList = driver.find_elements(By.CLASS_NAME, "tcg-breadcrumbs__list")[0].text
        ProductType = BreadCrumbList.split('\n')[1]
    except:
        'PK Site Not Found'
        ProductType = 'InvalidPK'
        FailedPKChecks = FailedPKChecks + 1

    #Check to see if product is a Pokemon card
    if ProductType == 'Pokemon Cards':

        # Split the BreadCrumbList
        tempList.append(BreadCrumbList.split('\n')[2])                                      #Series
        tempList.append(BreadCrumbList.split('\n')[3])                                      #CardName

        ############---------ItemDetails

        # Search for the ItemDetails
        ItemDetails = driver.find_elements(By.CLASS_NAME, "product__item-details__content")[0].text

        # Split the ItemDetails
        tempList.append(ItemDetails.split('\n')[1].split(' / ')[1].split(':')[1])           #CardNumber
        tempList.append(ItemDetails.split('\n')[1].split(' / ')[2])                         #Rarity

        ############---------SpotlightPrice

        # Search for the SpotlightPrice
        SpotlightPrice = driver.find_elements(By.CLASS_NAME, "spotlight__price")[0].text    #Removes the dollar sign
        tempList.append(SpotlightPrice[1:len(SpotlightPrice)])

        Extract.append(tempList)

# Close the browser
driver.quit()

ProductPK: 452021
ProductPK: 509983
ProductPK: 618701
ProductPK: 6187015555555


In [103]:
Extract

['ProductPK',
 'Series',
 'CardName',
 'CardNumber',
 'Rarity',
 'SpotlightPrice',
 ['452021',
  'SWSH12: Silver Tempest Trainer Gallery',
  'Rockruff',
  'TG07/TG30',
  'Ultra Rare',
  '2.97'],
 ['509983',
  'SV03: Obsidian Flames',
  'Pidgeot ex - 225/197',
  '225/197',
  'Special Illustration Rare',
  '13.52']]

In [25]:
#with open("MaxPK.txt", "r") as f:
#    MaxPK = int(f.readlines()[0])
MaxPK = 42346
LastSuccessfulPK = MaxPK
Extract = [] 
Headers = ['ProductPK','Series','CardName','CardNumber','Rarity','SpotlightPrice','InsertedDTM','LastModifiedDTM']
Extract.append(Headers)
BaseURL = 'https://www.tcgplayer.com/product/'

In [26]:
FailedPKChecks = 0
SearchMarker = 0
SearchLimit = 300

from selenium import webdriver
from selenium.webdriver.common.by import By
import time

CurrentDTM = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# Set up the WebDriver
driver = webdriver.Chrome()

while (SearchMarker < SearchLimit and FailedPKChecks < 100):
    
    print('[' + time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()) + '] ' + str(SearchMarker + 1) + ' out of ' + str(SearchLimit) +  ' - ProductPK: ' + str(MaxPK))

    # Open the website
    driver.get(BaseURL + str(MaxPK))

    tempList = []

    # Allow the page to load
    time.sleep(5)

    tempList.append(str(MaxPK))

    ############---------BreadCrumbList

    # Try searching the next PK
    try: 
        BreadCrumbList = driver.find_elements(By.CLASS_NAME, "tcg-breadcrumbs__list")[0].text
        ProductType = BreadCrumbList.split('\n')[1]
        LastSuccessfulPK = MaxPK
        FailedPKChecks = 0
    except:
        'PK Site Not Found'
        ProductType = 'InvalidPK'
        FailedPKChecks = FailedPKChecks + 1

    #Check to see if product is a Pokemon card
    if (ProductType == 'Pokemon Cards' or ProductType == 'Pokemon Japan'):

        # Split the BreadCrumbList
        tempList.append(BreadCrumbList.split('\n')[2])                                      #Series
        tempList.append(BreadCrumbList.split('\n')[3])                                      #CardName

        ############---------ItemDetails

        # Search for the ItemDetails
        ItemDetails = driver.find_elements(By.CLASS_NAME, "product__item-details__content")[0].text

        # Split the ItemDetails
        tempList.append(ItemDetails.split('\n')[1].split(' / ')[1].split(':')[1])           #CardNumber
        tempList.append(ItemDetails.split('\n')[1].split(' / ')[2])                         #Rarity

        ############---------SpotlightPrice

        # Search for the SpotlightPrice
        SpotlightPrice = driver.find_elements(By.CLASS_NAME, "spotlight__price")[0].text    #Removes the dollar sign
        tempList.append(SpotlightPrice[1:len(SpotlightPrice)])

        tempList.append(CurrentDTM)                                                         #InsertedDTM
        tempList.append(CurrentDTM)                                                         #LastModifiedDTM

        Extract.append(tempList)

    
    MaxPK = MaxPK + 1
    SearchMarker = SearchMarker + 1

# Close the browser
driver.quit()

with open("MaxPK.txt", "w") as f:
    f.write(str(LastSuccessfulPK))

[2025-05-31 01:22:11] 1 out of 300 - ProductPK: 42346
[2025-05-31 01:22:16] 2 out of 300 - ProductPK: 42347
[2025-05-31 01:22:22] 3 out of 300 - ProductPK: 42348
[2025-05-31 01:22:27] 4 out of 300 - ProductPK: 42349
[2025-05-31 01:22:32] 5 out of 300 - ProductPK: 42350
[2025-05-31 01:22:37] 6 out of 300 - ProductPK: 42351
[2025-05-31 01:22:42] 7 out of 300 - ProductPK: 42352
[2025-05-31 01:22:48] 8 out of 300 - ProductPK: 42353
[2025-05-31 01:22:53] 9 out of 300 - ProductPK: 42354
[2025-05-31 01:22:58] 10 out of 300 - ProductPK: 42355
[2025-05-31 01:23:03] 11 out of 300 - ProductPK: 42356
[2025-05-31 01:23:08] 12 out of 300 - ProductPK: 42357
[2025-05-31 01:23:14] 13 out of 300 - ProductPK: 42358
[2025-05-31 01:23:19] 14 out of 300 - ProductPK: 42359
[2025-05-31 01:23:24] 15 out of 300 - ProductPK: 42360
[2025-05-31 01:23:29] 16 out of 300 - ProductPK: 42361
[2025-05-31 01:23:34] 17 out of 300 - ProductPK: 42362
[2025-05-31 01:23:40] 18 out of 300 - ProductPK: 42363
[2025-05-31 01:23:4

In [ ]:
with open("MaxPK.txt", "w") as f:
    f.write(str(MaxPK))

In [15]:
#First is 42346
MaxPK

42627

In [35]:
Extract

[['ProductPK',
  'Series',
  'CardName',
  'CardNumber',
  'Rarity',
  'SpotlightPrice',
  'InsertedDTM',
  'LastModifiedDTM'],
 ['42346',
  'Base Set',
  'Alakazam',
  '001/102',
  'Holo Rare',
  '14.00',
  '2025-05-31 01:22:10',
  '2025-05-31 01:22:10'],
 ['42347',
  'Base Set',
  'Mewtwo',
  '010/102',
  'Holo Rare',
  '7.00',
  '2025-05-31 01:22:10',
  '2025-05-31 01:22:10'],
 ['42348',
  'Base Set',
  'Lightning Energy',
  '100/102',
  'Common',
  '0.15',
  '2025-05-31 01:22:10',
  '2025-05-31 01:22:10'],
 ['42349',
  'Base Set',
  'Psychic Energy',
  '101/102',
  'Common',
  '0.47',
  '2025-05-31 01:22:10',
  '2025-05-31 01:22:10'],
 ['42350',
  'Base Set',
  'Water Energy',
  '102/102',
  'Common',
  '0.34',
  '2025-05-31 01:22:10',
  '2025-05-31 01:22:10'],
 ['42351',
  'Base Set',
  'Nidoking',
  '011/102',
  'Holo Rare',
  '24.40',
  '2025-05-31 01:22:10',
  '2025-05-31 01:22:10'],
 ['42352',
  'Base Set',
  'Ninetales',
  '012/102',
  'Holo Rare',
  '19.99',
  '2025-05-31 01

In [36]:
len(Extract)

223

In [79]:
for row in Extract:
    print(','.join(str(item) for item in row ))

ProductPK,Series,CardName,CardNumber,Rarity,SpotlightPrice,InsertedDTM,LastModifiedDTM
42346,Base Set,Alakazam,001/102,Holo Rare,14.00,2025-05-31 01:22:10,2025-05-31 01:22:10
42347,Base Set,Mewtwo,010/102,Holo Rare,7.00,2025-05-31 01:22:10,2025-05-31 01:22:10
42348,Base Set,Lightning Energy,100/102,Common,0.15,2025-05-31 01:22:10,2025-05-31 01:22:10
42349,Base Set,Psychic Energy,101/102,Common,0.47,2025-05-31 01:22:10,2025-05-31 01:22:10
42350,Base Set,Water Energy,102/102,Common,0.34,2025-05-31 01:22:10,2025-05-31 01:22:10
42351,Base Set,Nidoking,011/102,Holo Rare,24.40,2025-05-31 01:22:10,2025-05-31 01:22:10
42352,Base Set,Ninetales,012/102,Holo Rare,19.99,2025-05-31 01:22:10,2025-05-31 01:22:10
42353,Base Set,Poliwrath,013/102,Holo Rare,32.83,2025-05-31 01:22:10,2025-05-31 01:22:10
42354,Base Set,Raichu,014/102,Holo Rare,31.98,2025-05-31 01:22:10,2025-05-31 01:22:10
42355,Base Set,Venusaur,015/102,Holo Rare,25.98,2025-05-31 01:22:10,2025-05-31 01:22:10
42356,Base Set,Zapdos,016/102,

In [80]:
with open("Extract.txt", "w") as f:
    for row in Extract:
        f.write(','.join(str(item) for item in row ) + '\n')

In [81]:
import csv

with open("Extract.txt", newline='\n') as f:
    reader = csv.reader(f)
    data = list(reader)

In [84]:
data[1]

['42346',
 'Base Set',
 'Alakazam',
 '001/102',
 'Holo Rare',
 '14.00',
 '2025-05-31 01:22:10',
 '2025-05-31 01:22:10']